In [1]:
%cd ~/code/entityrepresentations/

/home/morand/code/entityrepresentations


/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import transformers
import transformer_lens as tl 
from transformer_lens import HookedTransformer, patching
from importlib import reload
import torch, os, gc, sys, pathlib
import torch.nn as nn
from tqdm import tqdm
#our own code
import utils 
from IPython.display import display
# import ipywidgets as widgets
# reload(IPython.display)
# reload(widgets)
reload(utils)
from LabelExtractor import eval_model, infer_entities, evaluateTV
reload(sys.modules['LabelExtractor'])
from processResults import *
reload(sys.modules['processResults'])

Using the latest cached version of the module from /home/morand/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--chrf/d244bab9383988714085a8dacc4871986d9f025398581c33d6b2ee22836b4069 (last modified on Thu Jun 27 14:17:21 2024) since it couldn't be found locally at evaluate-metric--chrf, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from /home/morand/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--chrf/d244bab9383988714085a8dacc4871986d9f025398581c33d6b2ee22836b4069 (last modified on Thu Jun 27 14:17:21 2024) since it couldn't be found locally at evaluate-metric--chrf, or remotely on the Hugging Face Hub.


<module 'processResults' from '/home/morand/code/entityrepresentations/processResults.py'>

## Params 

In [3]:
# Load a model (eg GPT-2 Small)
model_name = "gpt2-small" # 117M ok
model_name = "pythia-2.8b"#ok !
model_name = "mistralai/Mistral-7B-v0.1" # ok JZ a100 8cpus | 
model_name = "gpt2-large" # 774M ok
model_name = "gpt2-medium" # 302M ok
model_name = "gpt2-xl" # 1.5B ok
model_name = "phi-1_5"  # 1.5B ok
model_name = "phi-2" # 2,5B ok 12cpus nope, gpu 24cpus ok

dataset_name= "WebNLG"
dataset_name= "TACRED"
dataset_name= "CoNLL2003"

dtype = torch.bfloat16
dtype = torch.float32

with_context = True
with_context = False

xp_path = pathlib.Path.home() / "experiments_JeanZay" # load results
xp_paths= os.listdir(xp_path / "jobs")      #get all experiments directories
jobs_path = xp_path / "jobs" / xp_paths[0]
results = loadResults(jobs_path)            #get jobs
results = results[(results["model_name"]==model_name) &
                  (results["dataset_name"]==dataset_name)
                #   (results["with_context"]==with_context) 
                  ]
print(f"Loaded {len(results)} results")
xp_lin = 'labelextractor.learnlinearfilter'
results_linear = loadResults(xp_path / "jobs" / xp_lin)
results_linear = results_linear[results_linear["Eval"].notna()]
results_linear["hash"] = results_linear["path"].apply(lambda x: x.parts[-1])
print(f"{len(results_linear)} linear jobs")
results.head()

100%|██████████████████████████████████████████████████████████████████████████| 2010/2010 [00:09<00:00, 210.54it/s]


Loaded 726 results


100%|████████████████████████████████████████████████████████████████████████████| 830/830 [00:03<00:00, 239.56it/s]


726 linear jobs


,path,model_name,layer,max_ent_length,epochs,lr,batch_size,with_context,dataset_name,extraction_method,history,Eval,date,inference,logs_per_epoch,max_length,run,version
60,/home/morand/experiments_JeanZay/jobs/labelext...,phi-2,31,20,10,0.04,30,True,CoNLL2003,in_context,"[{'epoch': 0.2490118577075099, 'loss': 1.51630...","{'Partial Match': 0.9218778994247542, 'Exact M...",1.726944e+09,/home/morand/experiments_JeanZay/jobs/labelext...,4.0,200.0,4.0,1.0
99,/home/morand/experiments_JeanZay/jobs/labelext...,phi-2,4,20,10,0.04,30,False,CoNLL2003,in_context,"[{'epoch': 0.2490118577075099, 'loss': 1.36120...","{'Partial Match': 0.6921506773056225, 'Exact M...",1.726855e+09,/home/morand/experiments_JeanZay/jobs/labelext...,4.0,200.0,0.0,1.0
145,/home/morand/experiments_JeanZay/jobs/labelext...,phi-2,28,20,10,0.04,30,False,CoNLL2003,in_context,"[{'epoch': 0.2490118577075099, 'loss': 1.46980...","{'Partial Match': 0.6574503618482093, 'Exact M...",1.726860e+09,/home/morand/experiments_JeanZay/jobs/labelext...,4.0,200.0,0.0,1.0
151,/home/morand/experiments_JeanZay/jobs/labelext...,phi-2,16,20,10,0.04,30,True,CoNLL2003,in_context,"[{'epoch': 0.2490118577075099, 'loss': 1.44305...","{'Partial Match': 0.9562070885136389, 'Exact M...",1.726929e+09,/home/morand/experiments_JeanZay/jobs/labelext...,4.0,200.0,4.0,1.0
154,/home/morand/experiments_JeanZay/jobs/labelext...,phi-2,6,20,10,0.04,30,False,CoNLL2003,in_context,"[{'epoch': 0.2490118577075099, 'loss': 1.50161...","{'Partial Match': 0.6863982185934311, 'Exact M...",1.726856e+09,/home/morand/experiments_JeanZay/jobs/labelext...,4.0,200.0,3.0,1.0


## Load Model

In [4]:
gc.collect()
torch.cuda.empty_cache()

In [5]:
# del model
#check if model variable exists
if not 'model' in locals():
    model = HookedTransformer.from_pretrained(
                                    model_name, 
                                    trust_remote_code = True, 
                                    low_cpu_mem_usage = True, 
                                    device_map='auto',
                                    move_to_device=False,
                                    fold_ln=False,
                                    fold_value_biases=False,
                                    center_writing_weights=False,
                                    center_unembed=False,
                                    )
    dim = model.QK.shape[-1]
print(model)
model.eval()
model = model.cuda()

/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loaded pretrained model phi-2 into HookedTransformer
HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out

## Load the data

In [6]:
demo_data_name = "TACRED"
demo_data_name = "WebNLG"
demo_data_name = "CoNLL2003"
train_dataset, test_dataset, dev_dataset = utils.load_datasets(demo_data_name)
print("test length:", len(test_dataset))
print("ex sample:", test_dataset[np.random.randint(len(test_dataset))])

test length: 5389
ex sample: {'entity': 'Behle', 'text': '1. Behle 89', 'id': 4070}


# Test TaskVecs

## Evaluation

In [7]:
#find all layers in results
layer = 20

b_size = 10
#augment test set with representations
print(f"augmenting test dataset with representations from layer {layer}")
test_dataset.augment_with_repr(model, layer, batch_size=b_size, method="in_context")

augmenting test dataset with representations from layer 20


100%|█████████████████████████████████████████████████████████████████████████████| 539/539 [01:34<00:00,  5.68it/s]


In [8]:

for with_context in  [False, True]:
    print(f"Comparing perf of {'contextual' if with_context else 'uncontextual'} taskVec trained at layer {layer} on {demo_data_name} dataset")
    #find all taskvecs in results
    fileName = get_taskVec(results, 
                        model_name, 
                        layer=layer, 
                        dataset_name=dataset_name, 
                        with_context = with_context
                        )
    TaskVec = torch.load(fileName)
    print("TaskVec loaded from ", str(fileName))

    # # compute metrics with and without context  
    # metrics = evaluateTV(  model,
    #                             TaskVec,
    #                             test_dataset,
    #                             b_size=10,
    #                             with_context=with_context,
    #                             )
    # print(f"Metrics on {'contextual' if with_context else 'uncontextual'} generation: ", metrics)
    
    # metrics = evaluateTV(  model,
    #                             TaskVec,
    #                             test_dataset,
    #                             b_size=10,
    #                             with_context= not with_context,
    #                             )
    # print(f"Metrics on {'contextual' if not with_context else 'uncontextual'} generation: ", metrics)

    # rand_TaskVec = torch.randn_like(TaskVec)
    # metrics = evaluateTV(  model,
    #                             rand_TaskVec,
    #                             test_dataset,
    #                             b_size=50,
    #                             with_context= not with_context,
    #                             )
    # print(f"Metrics on {'contextual' if not with_context else 'uncontextual'} generation with Random Task vector: ", metrics)

Comparing perf of uncontextual taskVec trained at layer 20 on CoNLL2003 dataset
found 11 jobs for layer 20 of phi-2 without context on CoNLL2003
found ['TaskVec_phi-2_l20_e10.0.pth'] 
TaskVec loaded from  /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/3320fdb2ffd15372a5a0b5a011871e86a696604b8335bfe874f2e363e1e91987/TaskVec_phi-2_l20_e10.0.pth
Comparing perf of contextual taskVec trained at layer 20 on CoNLL2003 dataset
found 11 jobs for layer 20 of phi-2 with context on CoNLL2003
found ['TaskVec_phi-2_l20_e10.0.pth'] 
TaskVec loaded from  /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/3988c4dfbc9e1cddee560ec79ac0bcb39cf869fd6d712532376f90a75642acca/TaskVec_phi-2_l20_e10.0.pth


## Compare with inference saved from Training

In [9]:
# with_context = False

# print(f"Comparing perf of {'contextual' if with_context else 'uncontextual'} taskVec trained at layer {layer} on {demo_data_name} dataset")
# #find all taskvecs in results
# fileName = get_taskVec(results, 
#                     model_name, 
#                     layer=layer, 
#                     dataset_name=dataset_name, 
#                     with_context = with_context
#                     )
# TaskVec = torch.load(fileName)
# # compute metrics with and without context  
# metrics = evaluateTV(  model,
#                             TaskVec,
#                             test_dataset,
#                             b_size=10,
#                             with_context=with_context,
#                             )
# print(f"Metrics on {'contextual' if with_context else 'uncontextual'} generation: ", metrics)

In [10]:
### Load inference file
inference = get_inference_res(results, model_name, layer, dataset_name, with_context, verbose=True)

for i, item in enumerate(inference):
    item['id'] = i
    item['inferred'] = item['generation']

##compute metrics
metrics = evaluateTV(  model,
                            TaskVec,
                            inference,
                            b_size=10,
                            with_context=with_context,
                            force_recompute=False
                            )
print(f"Metrics on RETRIEVED {'contextual' if with_context else 'uncontextual'} generation: ", metrics)

#print some fail examples
print("Ex of Failed examples:\n")
print("Original:".center(40) + " | " + "Inferred:".center(40)+ '\n'+"-"*84)
limit = 40
fails = set()
for item, gen in zip(inference, test_dataset):
    id = item['id']
    if len(fails) >= limit: break
    if item['entity'] == item['generation'] or item['generation'] in fails: continue
    fails |= {item['generation']}
    print(f"{item['entity'].center(40)} | {item['generation'].center(40)} | {gen['inferred'].center(40)}| {gen['entity'].center(40)}")

found 6 results for layer 20 of phi-2 with context on CoNLL2003
got inference files: /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/3988c4dfbc9e1cddee560ec79ac0bcb39cf869fd6d712532376f90a75642acca/Inference_phi-2_l20.json


100%|██████████████████████████████████████████████████████████████████████| 5389/5389 [00:00<00:00, 1214437.15it/s]


Metrics on RETRIEVED contextual generation:  {'Partial Match': 0.960103915383188, 'Exact Match': 0.9313416218222305, 'Chr-F': 93.88795018448242, 'Version': 1.2}
Ex of Failed examples:

               Original:                 |                Inferred:                
------------------------------------------------------------------------------------


KeyError: 'inferred'

In [19]:
#print some examples

print(f"stat|{'Entity'.center(40)}|{'Generation'.center(60)}|{'Context'.center(50)}")
print("="*190)

for i in range(15):
    item = test_dataset[np.random.randint(len(test_dataset))]
    ent = item['entity']
    gen = item['inferred'].replace('\n', "")
    success = "V" if ent.strip() == gen.strip() else "X"
    print(f" {success}  |{ent.center(40)}|{gen.center(60)}|  {item['text']}")


stat|                 Entity                 |                         Generation                         |                     Context                      
 X  |                 League                 |         The League of NationsThe League of Nations         |  League game on Thursday (home team in CAPS):
 X  |                 Havel                  |           ly, the first thing you need to do is            |  Havel had a small malignant tumour removed from his lung on Monday and is recovering in hospital.
 X  |                Linfield                |          The University of EdinburghEdinburgh, E           |  Linfield 10 4 3 3 13 10 15
 X  |                 LONDON                 |      The UK's leading independent source of news and       |  LONDON 1996-12-07
 X  |                 Japan                  |           Japan's new prime minister, Yoshihide            |  Japan's Economic Planning Agency has not changed its view that the economy is gradually recovering, despite r

# Demo

In [ ]:
from torch.utils.data import DataLoader
from torchmetrics.text import Perplexity

@torch.no_grad()
def infer_entities(model,
                   taskVector, 
                   dataset, 
                   with_context: bool = False, 
                   return_attn_pattern: bool = False,
                   return_ppx: bool = False,
                   return_likelihood: bool = False,
                   max_tokens = 20, 
                   b_size = 10, 
                   prepend_bos:bool = True
    ):
    """
    Infer entities from a given dataset augmented with representations.
    Will write the inferred entities back to the dataset in the 'inferred' field.
    Args:
        model: the model to use
        TaskVec: the task vector to use
        dataset: the dataset to infer entities from, 
            must have a 'representation' key that stores the entity representation to generate from. 
            must contain 'text' key for context
        with_context : bool : if True, the context is prepended to the entity
        return_attn_pattern : bool : if True, will return the attention pattern
        max_tokens: the maximum number of tokens to generate
        b_size: the batch size to use when inferring on a bug dataset
        prepend_bos: whether to prepend the BOS token to the input
    Returns:
        None, will write the inferred entities (and optionnally attn patterns) back to the dataset
    """
    assert taskVector is not None
    
    # inp_toks = model.to_tokens(, prepend_bos=prepend_bos)
    replace_hook_name = tl.utils.get_act_name('embed')
    #get the attention pattern hooks names
    pattern_hooks_names = [ tl.utils.get_act_name("pattern", l, "attn") for l in range(model.cfg.n_layers)]
    
    taskVector = taskVector.view(1,-1)
    dataloader = DataLoader(dataset, batch_size=b_size, shuffle=True)
    eos_tok = model.tokenizer.eos_token_id
    eos_tok_str = model.tokenizer.eos_token
    generated = []
    
    if return_ppx:
        perp = Perplexity(ignore_index=model.tokenizer.eos_token_id)
        perp = perp.cuda()

    #inference loop
    for batch in tqdm(dataloader):

        ids = batch["id"].detach().cpu().numpy()
        reps = batch["representation"].squeeze(1).cuda()
        b_size = reps.shape[0]
        b_taskVec = taskVector.repeat(b_size,1).cuda()
        
        if with_context:
            texts = batch["text"]
            prompts = [txt + "_ >" for txt in texts]
        else:
            prompts = ["_ >" for r in reps]

        inp_toks = model.to_tokens(prompts, prepend_bos=prepend_bos, padding_side="left") 
        rep_idx = inp_toks.shape[1] - 2
        taskVec_idx = rep_idx + 1
        rep_idxs = torch.tensor(b_size * [rep_idx])
        taskVec_idxs = torch.tensor(b_size * [taskVec_idx])
        fwd_hooks = [   
                        (replace_hook_name, utils.get_replace_with_rep_hook(reps, rep_idxs)), # replace '_' by the subject Representation
                        (replace_hook_name, utils.get_replace_with_rep_hook(b_taskVec, taskVec_idxs)) # replace 'called' by TaskVec Representation
                    ]
        #inference
        for i in range(max_tokens):
                # print(inputs, targets)
                logits = model.run_with_hooks(
                    inp_toks,
                    return_type = "logits",
                    fwd_hooks=fwd_hooks
                ,)

                final_logits =  logits[:,-1,:] #extract logits for last token only
                new_toks = final_logits.argmax(-1).view(-1,1)
                inp_toks = torch.hstack((inp_toks,new_toks))

                #check if we have reached the end of the sequence
                if all(new_toks == eos_tok): break

        if return_ppx or return_likelihood:
            # print(logits.shape, inp_toks.shape)
            for i in range(b_size):
                ppx = perp(
                    logits[i,:,:].unsqueeze(0), 
                    inp_toks[i,:-1].unsqueeze(0)
                    )
                dataset[ids[i]]["perplexity"] = ppx.item()
                #count non eos tokens
                n_toks = torch.sum(inp_toks[i,:] != eos_tok)
                #compute likelihood
                log_likelihood = torch.log(ppx) * n_toks
                dataset[ids[i]]["likelihood"] = log_likelihood.item()
                
        if return_attn_pattern:
            n_toks = inp_toks.shape[1]
            #instanciate the attention pattern
            attn_patterns = torch.zeros((   b_size,
                                            model.cfg.n_layers,
                                            model.cfg.n_heads,
                                            n_toks, 
                                            n_toks))
            def get_attn(
                pattern: torch.Tensor, # batch head_index dest_pos source_pos
                hook,
                ):
                """Hook function that stores the attention pattern"""
                l = int(hook.name.split(".")[1])
                attn_patterns[:,l,:,:,:] = pattern.cpu().detach()

            fwd_hooks += [ (hook, get_attn) for hook in pattern_hooks_names]
            #get the attention patterns for the batch
            model.run_with_hooks(
                    inp_toks,
                    return_type = None,
                    fwd_hooks=fwd_hooks
                ,)
            #store the attention patterns in the dataset
            for i in range(b_size):
                dataset[ids[i]]["attn_pattern"] = attn_patterns[i]

        for i in range(b_size) :
            gen = model.tokenizer.decode(
                inp_toks[i,taskVec_idxs[i]+1:].view(-1))
            # gen = "".join(gen).split(eos_tok_str)[0].strip()
            gen = gen.split(eos_tok_str)[0].strip()
            #store the inferred entity
            dataset[ids[i]]["inferred"] = gen
    return


### Load the taskVec

In [12]:
layer = 10
with_context = True
with_context = False
fileName = get_taskVec(results, 
                       model_name, 
                       layer=layer, 
                       dataset_name=dataset_name, 
                       with_context = with_context)

TaskVec = torch.load(fileName)
print("TaskVec loaded from ", fileName)

found 11 jobs for layer 10 of phi-2 without context on CoNLL2003
found ['TaskVec_phi-2_l10_e10.0.pth'] 
TaskVec loaded from  /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/2de398a401fb7a6de0f27cad4c60d7ffe5ab8046a5d0df1f9d032310e9bf9418/TaskVec_phi-2_l10_e10.0.pth


### Load Linear model

In [13]:

path = get_taskVec(results_linear, model_name, layer=layer, with_context=with_context)
print(f"Loading model from {str(path).split('/')[-1]}")
linear_filter = torch.load(path)
#cast the model to the correct type
linear_filter = linear_filter.type(dtype).cuda()


found 12 jobs for layer 10 of phi-2 without context 
found ['LinearFilter_phi-2_l10_e10.0noContext.pth'] 
Loading model from LinearFilter_phi-2_l10_e10.0noContext.pth


## Label-Lens

In [14]:
from circuitsvis.tokens import colored_tokens
prompt = "Locally nicknamed 'La dame de fer' (French for 'Iron Lady'), it was constructed as the centerpiece of the 1889 World's Fair, and to crown the centennial anniversary of the French Revolution. Although initially criticised by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognisable structures in the world. The tower received 5,889,000 visitors in 2022. The Eiffel Tower is the most visited monument with an entrance fee in the world"
prompt = "Bary Zito plays"
prompt = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. His job was"
prompt = "When Albert Einstein and his friend Mandelbrot went to the store, the father of relativity gave the bottle to"
prompt = train_dataset[np.random.randint(len(train_dataset))]["text"]

str_tokens = model.to_str_tokens(prompt)


In [15]:

#generation parameters
# with_context = True
# with_context = False
use_filter = False
use_filter = True

e_layer = 15
e_layer = layer

if use_filter: print("Using linear filter")

#get whole cache 
with torch.no_grad():
    _ , cache = model.run_with_cache(prompt)
    repr = cache[tl.utils.get_act_name("resid_post", e_layer, "")].detach().cpu() # 1 x n_tokens x dim
    del cache
    gc.collect()
    torch.cuda.empty_cache()

    data = [{ "id":i, 
            "representation": linear_filter(repr[0,i,:].cuda()).cpu() if use_filter else repr[0,i,:], 
            "text": prompt}
        for i in range(len(str_tokens)) ]

    #infer entities 
    print(f"infering entities from layer {e_layer} representations of all tokens in prompt with{'' if with_context else 'out'} context...")
    infer_entities(model,
                    TaskVec, 
                    data, 
                    with_context= with_context, 
                    max_tokens = 10, 
                    b_size = 5,
                    return_ppx = True,
                    )
    for d in data:
        d["logit_lens"] = utils.project_on_vocab(model,d["representation"], k=3)
    
    generated = [f"{d['inferred']} \n {d['logit_lens']}" for d in data]

html = colored_tokens(str_tokens,generated)
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 70px")
html


Using linear filter
infering entities from layer 10 representations of all tokens in prompt without context...


100%|█████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.23it/s]
/home/morand/code/entityrepresentations/utils.py:151: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  logits = torch.tensor(rep).float().cuda() @ model.W_U


## In a LogitLens Fashion

In [30]:
import plotly.graph_objects as go
import plotly.colors as pc
import numpy as np

def create_colored_table(data, strings, headers, row_headers, colorscale='Viridis', title= "Label_Lens", min_val=None, max_val=None):
    if min_val is None:
        min_val = np.min(data)
    if max_val is None:
        max_val = np.max(data)
    assert len(row_headers) == len(strings)


    # Normalize the data
    norm_data = (np.array(data) - min_val) / (2 * (max_val - min_val))

    # Generate colors using normalized data
    colors = [[pc.sample_colorscale(colorscale, norm_val, colortype='rgb')[0] for norm_val in row] for row in norm_data]

    #transpose the colors array
    headers = ['Layer'] + headers
    white = 'rgb(255,255,255)'
    colors = [[white] + colors_row for colors_row in colors]
    
    colors = np.transpose(colors)
    norm_data = np.transpose(norm_data)
    strings = np.transpose([[f'<b>{row_headers[i]}</b>'] + strings[i] for i in range(len(strings))])

    # Set the text colors, with a different color for the first row
    text_colors = ['black'] + ['black'] * (len(strings) - 1)  # Black for the first row, white for others
    CELL_HEIGHT = 60
    # Create the table
    fig = go.Figure(
        data=[go.Table(
        header=dict(
            values=[f'<b>{h}</b>' for h in headers],
            line_color='white', fill_color='white',
            align='center',font=dict(color='black', size=12)
        ),
        cells=dict(
            values=strings,
            fill_color=colors,
            align='center',
            font=dict(color=text_colors, size=12),
            height=CELL_HEIGHT
        ),
        #columnwidth=[85] * len(strings[0]),
    )])

    # Update layout
    fig.update_layout(
        title_text=title,
        height=len(strings[0]) * CELL_HEIGHT + 100,  # Adjust height based on number of rows
        width=len(strings) * 100,  # Adjust width based on number of columns
        margin=dict(l=20, r=20, t=50, b=10),
    )

    return fig


In [66]:
prompt = test_dataset[np.random.randint(len(test_dataset))]["text"]
prompt = "Somunkonwncitea is the capital of France, the capital of France is"
prompt = "Netherlands's capital is the city of"
prompt = "The City of Lights most iconic monument is the"
prompt = "When Albert Einstein and Mandelbrot went to the store, the General Relativity father gave the bottle to"
print("prompt:", prompt)

prompt: When Albert Einstein and Mandelbrot went to the store, the General Relativity father gave the bottle to


In [41]:
with_context = False
with_context = True

use_filter = True
use_filter = False

every_N = 4
str_tokens = model.to_str_tokens(prompt)
layers = list(range(0,model.cfg.n_layers, every_N))
print(f"Computing label-lens for layers {layers}")


#get whole cache 
with torch.no_grad():
    _ , cache = model.run_with_cache(prompt)
    reprs = []
    for layer in layers:
        if layer == 0:
            hook = tl.utils.get_act_name("embed")
        else:
            hook = tl.utils.get_act_name("resid_post", layer, "")
        reprs.append(cache[hook].detach().cpu()) # 1 x n_tokens x dim
    del cache
    gc.collect()
    torch.cuda.empty_cache()
data = [{ "representation": reprs[i][0,t,:], "layer": l, "text": prompt}
    for t in range(len(str_tokens))
    for i,l in enumerate(layers) ]

if use_filter:
    for d in data:
        d["representation"] = linear_filter(d["representation"].cuda()).cpu()

#add the id
for i,d in enumerate(data):
    d["id"] = i

#infer entities 
print(f"infering entities from all tokens in prompt with{'' if with_context else 'out'} context...")
infer_entities( model,
                TaskVec, 
                data, 
                with_context= with_context, 
                max_tokens = 8, 
                b_size = 5,
                return_ppx = True,
                return_likelihood = True,
                )

#generate the table
strings = []
ppx = []
log_likelihoods = []
max_val = max([d["perplexity"] for d in data])
for l in layers:
    strings.append([d["inferred"] for d in data if d["layer"] == l])
    ppx.append([d["perplexity"] for d in data if d["layer"] == l])
    log_likelihoods.append([d["likelihood"] for d in data if d["layer"] == l])


Computing label-lens for layers [0, 4, 8, 12, 16, 20, 24, 28]
infering entities from all tokens in prompt with context...


100%|███████████████████████████████████████████████████████████████████████████████| 37/37 [00:26<00:00,  1.40it/s]


In [67]:
with_context = False
with_context = True

use_TV_layer = 20
use_TV_layer = None

print(f"Computing {'contextual' if with_context else 'uncontextual'} Entity Lens")



use_filter = True
use_filter = False

every_N = 5
str_tokens = model.to_str_tokens(prompt)

layers = [-1] + list(range(0,model.cfg.n_layers, every_N))

print(f"Computing label-lens for layers {layers}")


#get whole cache 
with torch.no_grad():
    _ , cache = model.run_with_cache(prompt)
    reprs = []
    for layer in layers:
        if layer == -1:
            hook = tl.utils.get_act_name("embed")
        else:
            hook = tl.utils.get_act_name("resid_post", layer, "")
        reprs.append(cache[hook].detach().cpu()) # 1 x n_tokens x dim
    del cache
    gc.collect()
    torch.cuda.empty_cache()
data = []

print(reprs[0].shape)

def get_TV(layer):
    fileName = get_taskVec(results, 
                        model_name, 
                        layer=layer, 
                        dataset_name=dataset_name, 
                        with_context = with_context,
                        verbose=False
                        )
    print(f" layer {layer}, TaskVec loaded from {fileName}")
    return torch.load(fileName)

if use_TV_layer is not None:
    TaskVec = get_TV(use_TV_layer)
    print(f"Using TaskVec from layer {use_TV_layer}")
else:
    print("will load TaskVec from every layer")

print(f"infering entities from all tokens in prompt with{'' if with_context else 'out'} context...")
for i, l in enumerate(layers):
    #load the task vector
    if use_TV_layer is None:
        TaskVec = get_TV(l)

    data_l = [{ "representation": reprs[i][0,t,:], "layer": l, "text": prompt}
        for t in range(len(str_tokens))
        ]

    if use_filter:
        path = get_taskVec(results_linear, model_name, layer=layer, with_context=with_context)
        print(f"Loading Linear Model from {str(path).split('/')[-1]}")
        linear_filter = torch.load(path)
        #cast the model to the correct type
        linear_filter = linear_filter.type(dtype).cuda()

    #add the id
    for i,d in enumerate(data_l):
        d["id"] = i

    #infer entities 
    infer_entities( model,
                    TaskVec, 
                    data_l, 
                    with_context= with_context, 
                    max_tokens = 8, 
                    b_size = 5,
                    return_ppx = True,
                    return_likelihood = True,
                    )
    data += data_l

#generate the table
strings = []
ppx = []
log_likelihoods = []
max_val = max([d["perplexity"] for d in data])
for l in layers:
    strings.append([d["inferred"] for d in data if d["layer"] == l])
    ppx.append([d["perplexity"] for d in data if d["layer"] == l])
    log_likelihoods.append([d["likelihood"] for d in data if d["layer"] == l])


Computing contextual Entity Lens
Computing label-lens for layers [-1, 0, 5, 10, 15, 20, 25, 30]
torch.Size([1, 23, 2560])
will load TaskVec from every layer
infering entities from all tokens in prompt with context...
 layer -1, TaskVec loaded from /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/3393169f8114b47ff3e41d26fd37d3dee2cedd1138f7a69ff29a69a6b9f07e70/TaskVec_phi-2_l-1_e10.0.pth


100%|█████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.97it/s]


 layer 0, TaskVec loaded from /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/0ef0b442478ed34e55947bf60dffb12473edfad2a9c9e7604edce89cf6b163b8/TaskVec_phi-2_l0_e10.0.pth


100%|█████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.63it/s]


 layer 5, TaskVec loaded from /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/1f74c3e0a6268acbaed19ed748be145c4b66d112fbb249da27f55a5d44395741/TaskVec_phi-2_l5_e10.0.pth


...

100%|█████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.46it/s]


 layer 25, TaskVec loaded from /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/66cf00ea418961973508a4cc19e8f78380b786beafb67bdc2bbdd5744e297f81/TaskVec_phi-2_l25_e9.996047430830039.pth


100%|█████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.97it/s]


 layer 30, TaskVec loaded from /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/078a8c5aa98a18eacd2dfa22743d909ee4906f2c83f03c468b58c1552cb8b09e/TaskVec_phi-2_l30_e10.0.pth


100%|█████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.35it/s]


In [68]:
title = f"{'Uncontextual' if not with_context else 'Contextual'} Entity-Lens"
if use_TV_layer is not None:
    suptitle = f", TaskVec from layer {use_TV_layer}"
else:
    suptitle = ", TaskVec from all layers"

fig = create_colored_table(
                        # ppx,
                        log_likelihoods, 
                        
                        strings, 
                        headers=str_tokens, 
                        
                        row_headers= [f"{l+1}" for l in layers],
                        colorscale="blues", 
                        title=title + suptitle,
                        # min_val=0, max_val=max_val
                        )
#save as pdf and show
fig.write_image(f"plots/{title.replace(' ','_')}.pdf")
fig.show()

In [40]:

fig = create_colored_table(
                        ppx,
                        # likelihood, 
                        strings, 
                        headers=str_tokens, 
                        row_headers= [f"layer {l}" for l in layers],
                        colorscale="blues", min_val=0, max_val=max_val)
fig.show()